In [29]:
from pathlib import Path
from pyteomics import mzid
import json
import re
from usigrabber.main import download_ftp
from usigrabber.utils import project_root_path
from typing import Any, Callable, TypedDict, Optional
from collections import defaultdict

In [7]:
SAMPLE_ACCESSION = "PXD001357"
usigrabber_root = project_root_path()
root_path = usigrabber_root / "data" / "project_archive"
project_path = root_path / SAMPLE_ACCESSION

filepath = project_path / "OTE0019_York_060813_JH16_F119502.mzid"

In [8]:
if not project_path.exists():
    download_ftp(
        "ftp://ftp.pride.ebi.ac.uk/pride/data/archive/2015/02/PXD001357/OTE0019_York_060813_JH16_F119502.mzid.gz",
        project_path,
    )

Saved to /home/gwauge/Documents/uni/master/mp/usiGrabber/data/project_archive/PXD001357/OTE0019_York_060813_JH16_F119502.mzid.gz


In [30]:
class Mod(TypedDict):
    location: int
    residue: Optional[str]
    modification: str

Mods = list[Mod]

def splice_mods(seq: str, mods: Mods, *, index_base: int = 1, wrap: Callable[[str], str] = lambda m: f"[{m}]"):
    """
    Insert each modification string into `seq` AFTER the residue at `location`.

    Args:
        seq (str): original sequence, e.g. "ABCDEF"
        mods (list[dict]): keys {"location": int, "residue": str|None, "modification": str}
            - Default indexing is 1-based (index_base=1).
            - Special case (1-based only): location == 0 with residue None inserts at start.
        index_base (int): 1 (default) or 0. Positions are into the ORIGINAL seq.
        wrap (callable): renderer for a modification; default "[Mod]".
    Returns:
        str: sequence with modifications spliced in.
    """
    # Group modifications by normalized 0-based location.
    # Use sentinel -1 for "before the first residue".
    grouped = defaultdict(list)

    for m in mods:
        loc = m["location"]
        res = m.get("residue", None)
        mod = m["modification"]

        if index_base == 1:
            if loc == 0:
                # only allowed when residue is None
                if res not in (None, ""):
                    raise ValueError("location 0 must have no residue (use None or omit).")
                grouped[-1].append(mod)
                continue
            loc0 = loc - 1
        elif index_base == 0:
            if loc < 0:
                raise IndexError("location must be >= 0 for 0-based indexing")
            loc0 = loc
        else:
            raise ValueError("index_base must be 0 or 1")

        if not (0 <= loc0 < len(seq)):
            raise IndexError(f"location {loc} out of range for sequence of length {len(seq)} (index_base={index_base})")

        # Sanity check residue if provided
        if res is not None and res != seq[loc0]:
            raise ValueError(f"Residue mismatch at location {loc}: expected '{seq[loc0]}', got '{res}'")

        grouped[loc0].append(mod)

    out = []
    cursor = 0

    # Emit any beginning-of-sequence mods first (sentinel -1), if present
    if -1 in grouped:
        out.extend(wrap(m) for m in grouped[-1])
        # then fall through to normal stitching

    # Stitch once over the ORIGINAL sequence, inserting AFTER each residue
    for loc0 in sorted(k for k in grouped.keys() if k != -1):
        out.append(seq[cursor:loc0 + 1])                 # include the residue at loc0
        out.extend(wrap(m) for m in grouped[loc0])       # then insert its mods
        cursor = loc0 + 1

    out.append(seq[cursor:])  # tail
    return "".join(out)

In [33]:
seq = "ABCDEF"

# If your locations are 1-based (common in bio) for the same visual result:
mods_1based: Mods = [
    {"location": 0, "residue": None, "modification": "Mod0"},
    {"location": 3, "residue": "C", "modification": "Mod1"},
    {"location": 5, "residue": "E", "modification": "Mod2"},
]
print(splice_mods(seq, mods_1based, index_base=1))
# -> "ABC[Mod1]DE[Mod2]F"


[Mod0]ABC[Mod1]DE[Mod2]F


In [10]:
obj = mzid.MzIdentML(source=str(filepath))

In [ ]:
counter = 0
for item in mzid.read(source=str(filepath)):
    counter += 1

    # if item.get("spectrumID") == "index=3809":
    #     print(item)
    #     print(json.dumps(item, indent=4))

    title = item.get("spectrum title")
    pattern = re.compile(r'File:"(?P<filename>[^"]+)"[\s\S]*?scan=(?P<scan>\d+)')
    m = pattern.search(title)
    if not m:
        print(f"WARNING: No match for {item.get('spectrumID')} with title {title}")
        continue

    filename = Path(m.group('filename')).stem
    scan_number = int(m.group('scan'))

    if len(item["SpectrumIdentificationItem"]) > 1:
        print(f"WARNING: Multiple SpectrumIdentificationItem for {item.get('spectrumID')}")

    seq_obj: dict[str, Any] = item["SpectrumIdentificationItem"][0]
    charge = seq_obj.get("chargeState")
    seq: list[str] = seq_obj.get("PeptideSequence", "").split()
    if seq_obj.get("Modification"):
        inserted = 0
        for mod in seq_obj["Modification"]:
            loc = mod.get("location") + inserted
            if loc > 1:
                loc -= 1  # shift for 1-based indexing
            name = mod.get("name")
            # insert modification notation into sequence
            seq.insert(loc, f"[{name}]")
            inserted += 1

        usi = f"mzspec:{SAMPLE_ACCESSION}:{filename}:scan:{scan_number}:{''.join(seq)}/{charge}"

        if len(seq_obj.get("Modification")) > 1:
            print(f"raw sequence: {seq_obj.get('PeptideSequence')}")
            print(json.dumps(seq_obj.get("Modification"), indent=4))
            print(usi)

print(f"Total items: {counter}")

{'spectrumID': 'index=3809', 'SpectrumIdentificationItem': [{'calculatedMassToCharge': 318.1792045, 'chargeState': 2, 'experimentalMassToCharge': 318.178955, 'rank': 2, 'passThreshold': False, 'PeptideEvidenceRef': [{'start': 435, 'end': 439, 'pre': 'S', 'post': 'G', 'isDecoy': False, 'PeptideSequence': 'VFVNR', 'Modification': [{'location': 4, 'residues': ['N'], 'monoisotopicMassDelta': 0.984016, 'name': 'Deamidated'}], 'accession': 'elim_c_1_5613', 'protein description': 'elim_c_1_5613', 'location': 'file:////databases/mascot/sequence/OralMicrobiome/current/OralMicrobiome20121013.fasta', 'name': 'OralMicrobiome', 'numDatabaseSequences': 4184993, 'numResidues': 737477549, 'version': 'OralMicrobiome20121013.fasta', 'FileFormat': 'FASTA format', 'DatabaseName': {'OralMicrobiome20121013.fasta': ''}, 'database type amino acid': ''}], 'Mascot:score': 18.94, 'Mascot:expectation value': 0.499168422906294, 'peptide unique to one protein': '', 'PeptideSequence': 'VFVNR', 'Modification': [{'loc